In [1]:
import numpy as np
import plotly.subplots as sp
import plotly.graph_objects as go
from manim import *

# Euler's Method

$$ x_{n+1} = x_{n} + \Delta t \bigg(\frac{dx}{dt}\bigg)_n $$

In [2]:
def eulers_method(state, step_size, derivative_function, *derivative_function_args):
    return state + (step_size * derivative_function(state, *derivative_function_args ))

In [3]:
def eulers_improved_method(state, step_size, derivative_function, *derivative_function_args):
    k1 = derivative_function(state, *derivative_function_args)
    trial = state + (step_size * k1)
    k2 = derivative_function(trial, *derivative_function_args)

    return (state + (step_size/2 * (k1 + k2)))

# Derivatives

In [4]:
def derivatives(state, spring_constant, mass):
    x_position, velocity = state

    return np.array([velocity, (-spring_constant * x_position) / mass])

# Mass-Spring Physics

In [5]:
dt = 0.01 # step size of 0.05s
k = 2 # spring constant of 10 Nm^-1
m = 2 # mass of 2kg
T = 2 * np.pi * np.sqrt(m/k) # time period of 1 oscillation
t_end = 2 * T # simulation runs for 10 time periods

t = np.arange(0, t_end + dt, dt) # each time step

## Using Euler's Method

In [6]:
x_standard = np.zeros(len(t)) # x positions
v_standard = np.zeros(len(t)) # velocities

# initial conditions
x_standard[0] = 0
v_standard[0] = 2

for i in np.arange(0, len(t) - 1, 1): 
    state = eulers_method(np.array([x_standard[i], v_standard[i]]), dt, derivatives, k, m)
    
    x_standard[i + 1] = state[0]
    v_standard[i + 1] = state[1]

E_standard = 1/2 * (m * v_standard**2 + k * x_standard**2) # total energy in the system

## Using Euler's Improved Method

In [7]:
x_improved = np.zeros(len(t)) # x positions
v_improved = np.zeros(len(t)) # velocities

# initial conditions
x_improved[0] = 0
v_improved[0] = 2

for i in np.arange(0, len(t) - 1, 1): 
    state = eulers_improved_method(np.array([x_improved[i], v_improved[i]]), dt, derivatives, k, m)

    x_improved[i + 1] = state[0]
    v_improved[i + 1] = state[1]

E_improved = 1/2 * (m * v_improved**2 + k * x_improved**2) # total energy in the system


## Plot of Displacement and Energy

In [8]:
fig = sp.make_subplots(
    rows=2,
    cols=1, 
    subplot_titles=("Displacement", "Total Energy")
)

# Displacement 
fig.add_trace(go.Scatter(x=t, y=x_standard, name="Standard Displacement"), row=1, col=1)
fig.add_trace(go.Scatter(x=t, y=x_improved, name="Improved Displacement"), row=1, col=1)


# Energy
fig.add_trace(go.Scatter(x=t, y=E_standard, name="Standard Energy"), row=2, col=1)
fig.add_trace(go.Scatter(x=t, y=E_improved, name="Improved Energy"), row=2, col=1)

fig.update_xaxes(title_text="Time / s", col=1)
fig.update_yaxes(title_text="Displacement / m", col=1, row=1)
fig.update_yaxes(title_text="Energy / J", col=1, row=2)

fig.update_layout(
    title=f"Displacement and energy for a mass, {m}kg, oscillating on a spring, spring constant {k}Nm<sup>-1</sup>",
    template="plotly_dark",
    height=800
)

fig.show()

# Simulation

In [9]:
%%manim -qh -v WARNING MassSpringScene

class MassSpringScene(Scene):
    def construct(self):
        x = x_improved

        wall = LEFT * 4 # point where spring attaches

        mass = Dot(point=wall + RIGHT * x[0])

        spring = Line(wall, mass.get_center())

        tracker = ValueTracker(0)

        mass.add_updater(lambda m: m.move_to(
            wall + RIGHT * (2 + x[int(tracker.get_value())]) # move mass to x values
        ))

        spring.add_updater(lambda s: s.put_start_and_end_on(
            wall, mass.get_center()
        )) # adjust spring according to mass position

        self.add(mass, spring)
        self.play(tracker.animate.set_value(len(x) - 1), run_time=t_end, rate_func=linear) # play animation at a linear rate

Manim Community v0.20.1

# Comments

Using Euler's method artificially introduces a energy into the system with each step, hence the amplitude of displacement of the mass from equilibrium is continually increasing. Using Euler's improved method significantly reduces this artificial energy, giving a more accurate simulation of the oscillations.